# Bayesian Model Comparison

Compare local-evidence classifiers on the same parent-text partitions and cached windows.

In [ ]:
from __future__ import annotations

import numpy as np
import os
from pathlib import Path

from IPython.display import display
import pandas as pd

from local_evidence_comparison_utils import (
    plot_model_score_heatmap,
    plot_normalized_token_metrics_with_model_evidence,
    select_comparison_texts,
)
from local_evidence_models_utils import (
    BayesianConfig,
    GaussianMixtureNBConfig,
    LOCAL_FEATURE_COLUMNS,
    MODEL_NAMES,
    run_local_evidence_model_comparison,
)
from local_evidence_utils import (
    default_local_evidence_path,
    load_or_build_local_evidence_windows,
    split_local_evidence_texts,
)
from token_candidate_similarity_heatmaps_utils import (
    load_similarity_row,
    load_text_row,
    plot_normalized_token_metrics_with_local_evidence,
    text_index_for_id,
    validated_text_summary,
    visualize_text_similarities,
)

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
PROCESSED_ROOT = DATA_ROOT / "processed"
DATASET = "kaggle_train"
MODEL = "opt_350m"
WINDOW_LENGTHS = (32, 64, 128)
MODEL_NAMES = ("logistic_regression", "gaussian_nb", "gaussian_mixture_nb", "pymc_bayesian_logistic")
LOCAL_EVIDENCE_MODEL = "logistic_regression"
STRIDE = 16
TOP_K = 20
SMOOTHING_WINDOW = 10
SEED = 42
ADVI_ITERATIONS = 10_000
POSTERIOR_DRAWS = 1_000
SELECTED_TEXTS = ()
RECOMPUTE_WINDOWS = False
try:
    import pymc
except ModuleNotFoundError:
    PYMC_AVAILABLE = False
else:
    PYMC_AVAILABLE = True
if not PYMC_AVAILABLE:
    display("PyMC unavailable; continuing with LR, GaussianNB, and GaussianMixtureNB. Install it with: uv sync --extra bayesian")
AVAILABLE_MODELS = tuple(name for name in MODEL_NAMES if name != "pymc_bayesian_logistic" or PYMC_AVAILABLE)
BAYESIAN_CONFIG = BayesianConfig(
    advi_iterations=ADVI_ITERATIONS,
    posterior_draws=POSTERIOR_DRAWS,
    random_seed=SEED,
)
GAUSSIAN_MIXTURE_COMPONENTS = 2
GAUSSIAN_MIXTURE_CONFIG = GaussianMixtureNBConfig(
    components=GAUSSIAN_MIXTURE_COMPONENTS,
    reg_covar=1e-6,
)
pd.set_option("display.max_columns", 100)

In [ ]:
WINDOWS_PATH = default_local_evidence_path(PROCESSED_ROOT, DATASET, MODEL)
windows = load_or_build_local_evidence_windows(
    PROCESSED_ROOT,
    DATASET,
    MODEL,
    path=WINDOWS_PATH,
    window_lengths=WINDOW_LENGTHS,
    stride=STRIDE,
    top_k=TOP_K,
    recompute=RECOMPUTE_WINDOWS,
)
display(windows.groupby(["split", "window_length"]).size().rename("window_rows").reset_index())

In [ ]:
partitions = split_local_evidence_texts(windows, seed=SEED)
comparison_summary, comparison_predictions, scored_windows, coefficient_summaries = run_local_evidence_model_comparison(
    windows,
    partitions,
    AVAILABLE_MODELS,
    feature_columns=LOCAL_FEATURE_COLUMNS,
    seed=SEED,
    bayesian_config=BAYESIAN_CONFIG,
    gaussian_mixture_config=GAUSSIAN_MIXTURE_CONFIG,
)
display(
    comparison_summary.sort_values(["model", "window_length", "aggregation"])[
        ["model", "window_length", "aggregation", "accuracy", "macro_f1"]
    ].reset_index(drop=True)
)
display(coefficient_summaries)
display(comparison_summary.groupby(["model", "window_length", "aggregation"]).size())

In [ ]:
test_predictions = comparison_predictions.query('partition == "test"').copy()
if SELECTED_TEXTS:
    selected_predictions = test_predictions.loc[test_predictions["text_id"].isin(SELECTED_TEXTS)]
else:
    selected_predictions = select_comparison_texts(test_predictions, count=3, seed=SEED)
display(selected_predictions)
display(
    selected_predictions.assign(
        is_error=lambda frame: frame["prediction"].ne(frame["label"])
    ).loc[:, ["model", "window_length", "aggregation", "text_id", "label", "prediction", "is_error"]]
)

In [ ]:
for _, selected in selected_predictions.drop_duplicates(["split", "text_id"]).iterrows():
    text_id = int(selected["text_id"])
    split = selected["split"]
    text_index = text_index_for_id(PROCESSED_ROOT, DATASET, MODEL, split, text_id)
    text_row = load_text_row(PROCESSED_ROOT, DATASET, MODEL, split, text_index)
    similarity_row = load_similarity_row(PROCESSED_ROOT, DATASET, MODEL, split, text_id)
    token_summary, token_labels = validated_text_summary(text_row, similarity_row, top_k=TOP_K)
    for window_length in WINDOW_LENGTHS:
        text_windows = scored_windows.loc[
            scored_windows["partition"].eq("test")
            & scored_windows["split"].eq(split)
            & scored_windows["text_id"].eq(text_id)
            & scored_windows["window_length"].eq(window_length)
        ]
        if text_windows.empty:
            continue
        local_evidence_windows = text_windows.loc[
            text_windows["model"].eq(LOCAL_EVIDENCE_MODEL)
        ]
        if local_evidence_windows.empty:
            raise ValueError(f"selected local evidence model is unavailable: {LOCAL_EVIDENCE_MODEL}")
        display(plot_model_score_heatmap(text_windows, len(token_labels), f"Model scores: text {text_id}, window {window_length}"))
        display(plot_normalized_token_metrics_with_model_evidence(
            token_summary, text_windows, len(token_labels), window=SMOOTHING_WINDOW
        ))
        display(f"Local evidence model: {LOCAL_EVIDENCE_MODEL}")
        display(plot_normalized_token_metrics_with_local_evidence(
            token_summary, local_evidence_windows, len(token_labels), window=SMOOTHING_WINDOW
        ))
        visualize_text_similarities(
            PROCESSED_ROOT, DATASET, MODEL, split, text_index, top_k=TOP_K,
            token_range=(int(text_windows["start_token"].min()), int(text_windows["stop_token"].max())),
        )
        finite_intervals = text_windows.loc[
            np.isfinite(text_windows["mg_score_lower"])
            & np.isfinite(text_windows["mg_score_upper"])
        ]
        if not finite_intervals.empty:
            display(f"PyMC posterior predictive bands available for text {text_id}")